# Polarscope - Complete API Reference Demo

This notebook demonstrates **every public function** in the `polarscope` library,
showing **every argument** and **every valid option** for each argument.

| # | Function | Purpose |
|---|----------|---------|
| 1 | `datasets` | Built-in dataset loaders and helpers |
| 2 | `xray()` | Data inspection & quality assessment |
| 3 | `clean_column_names()` | Column name standardisation |
| 4 | `convert_datatypes()` | Memory-efficient dtype optimisation |
| 5 | `drop_missing()` | Missing-value removal |
| 6 | `data_cleaning()` | Full cleaning pipeline |
| 7 | `corr_heatmap()` | Correlation heatmap |
| 8 | `dist_plot()` | Distribution histogram |
| 9 | `missingval_plot()` | Missing-value bar chart |
| 10 | `cat_plot()` | Categorical frequency plot |
| 11 | `corr_plot()` | Enhanced correlation plot |
| 12 | `save_fig()` | Universal figure export |

In [1]:
# Install / upgrade polarscope from the local development source
# This ensures the demo always runs against the latest local code.
%pip install -e "..[all]" --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# 0 | SETUP
# ============================================================
import polars as pl
import numpy as np
import polarscope as ps

print(f"Package : {ps.__title__}")
print(f"Version : {ps.__version__}")
print(f"Desc    : {ps.__description__}")

Package : polarscope
Version : 1.3.14
Desc    : 🔬 Simple data inspection tools for Polars


---
## 1 | Built-in Datasets

| Function | Description |
|----------|-------------|
| `titanic()` / `load_titanic()` | Titanic passenger data (156 rows, 12 cols, has nulls) |
| `diabetes()` / `load_diabetes()` | Pima diabetes data (768 rows, 9 cols, all numeric) |
| `list_datasets()` | Returns list of available dataset names |
| `dataset_info(name)` | Returns metadata string about a dataset (`name='titanic'\|'diabetes'`) |

In [3]:
from polarscope.datasets import (
    titanic, diabetes,            # convenience aliases
    load_titanic, load_diabetes,   # explicit loaders
    list_datasets, dataset_info,   # helpers
)

# list_datasets() -> returns ['titanic', 'diabetes']
print("Available:", list_datasets())
print()

# dataset_info(name) -> name options: 'titanic' | 'diabetes'
print(dataset_info('titanic'))
print()
print(dataset_info('diabetes'))

Available: ['titanic', 'diabetes']

Titanic Dataset
Famous passenger manifest from RMS Titanic with survival outcomes.

• Rows: 156 passengers
• Columns: 12 features  
• Missing values: Yes (Age, Cabin, Embarked)
• Use case: Classification, missing value analysis, categorical data
• Perfect for: Demonstrating xray(), missing value plots, correlation analysis

Diabetes Dataset  
Pima Indians Diabetes medical diagnostic data.

• Rows: 768 patients
• Columns: 9 features
• Missing values: None (but zeros may represent missing)
• Use case: Medical prediction, statistical analysis
• Perfect for: Demonstrating statistical tests, distribution analysis, correlation


In [4]:
# Load titanic dataset as a Polars DataFrame
df_titanic = titanic()

print(f"Titanic shape: {df_titanic.shape}")
df_titanic.head(3)

Titanic shape: (156, 12)


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""


In [5]:
# Load diabetes dataset as a Polars DataFrame
df_diabetes = diabetes()

print(f"Diabetes shape: {df_diabetes.shape}")
df_diabetes.head(3)

Diabetes shape: (768, 9)


Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
i64,i64,i64,i64,i64,f64,f64,i64,i64
6,148,72,35,0,33.6,0.627,50,1
1,85,66,29,0,26.6,0.351,31,0
8,183,64,0,0,23.3,0.672,32,1


---
## 2 | `xray()` - The Core Inspection Function

### Full signature
```python
ps.xray(
    df,
    *,
    include            = None,          # None | 'all' | 'numeric' | 'string' | 'temporal' | ['Float64','String',...]
    great_tables       = True,          # True -> GT table | False -> pl.DataFrame
    expanded           = False,         # True -> all stats | False -> essential stats only
    title              = None,          # str | None - custom title for GT output
    percentiles        = None,          # list[float] | None - default [0.25, 0.5, 0.75]
    outlier_method     = 'iqr',         # 'iqr' | 'percentile' | 'zscore'
    outlier_bounds     = None,          # [lower, upper] - required when outlier_method='percentile'
    corr_target        = None,          # str | None - numeric column name for correlation
    normality_test     = 'shapiro',     # 'shapiro' | 'anderson' | 'ks'  (expanded mode only)
    uniformity_test    = 'ks',          # 'ks' | 'chi2'                   (expanded mode only)
    missing_threshold  = 0.3,           # float - flag columns with > this fraction missing
    constant_threshold = 0.99,          # float - flag quasi-constant columns
    skew_threshold     = 2.0,           # float - flag |skew| above this
    kurtosis_threshold = 7.0,           # float - flag |kurtosis| above this
    outlier_threshold  = 0.05,          # float - flag outlier % above this
    shakiness_threshold= 2,             # int   - minimum score for SHAKY flag
    model_usability    = False,         # True -> add Usability_Flags, Usability_Score, Recommendation
    distribution_plot  = 'histogram',   # 'histogram'
    decimals           = 2,             # int - decimal places in GT numbers
    sep_mark           = ',',           # str - thousands separator (e.g. ',' or ' ')
    dec_mark           = '.',           # str - decimal mark (e.g. '.' or ',')
    compact            = False,         # True -> '10K' instead of '10,000'
    pattern            = None,          # str | None - e.g. '({x})' wraps each value
)
```

### 2.1 Default call (numeric columns only, GT table)

In [6]:
# Defaults: include=None, great_tables=True, expanded=False
# Shows only numeric columns with essential stats
ps.xray(df_titanic)

GT(_tbl_data=shape: (7, 16)
┌─────────────┬─────────┬───────┬────────────┬───┬─────────────┬────────────┬────────┬─────────────┐
│ Column      ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Pct_Missing ┆ N_Outliers ┆ skew   ┆ Distributio │
│ ---         ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---         ┆ ---        ┆ ---    ┆ n_Plot      │
│ str         ┆ str     ┆ i64   ┆ i64        ┆   ┆ f64         ┆ i64        ┆ f64    ┆ ---         │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ list[f64]   │
╞═════════════╪═════════╪═══════╪════════════╪═══╪═════════════╪════════════╪════════╪═════════════╡
│ PassengerId ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.0    ┆ [13.0,      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 13.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 13.0]       │
│ Survived    ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.647  ┆ [102.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, …      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 54.0]       │
│ Pclass      ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ -0.904 ┆ [30.0, 0.0, │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 96.0]     │
│ Age         ┆ Float64 ┆ 126   ┆ 30         ┆ … ┆ 19.23       ┆ 4          ┆ 0.692  ┆ [9.0, 4.0,  │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 4.0]      │
│ SibSp       ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 12         ┆ 2.199  ┆ [98.0, 0.0, │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 2.0]      │
│ Parch       ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 35         ┆ 2.734  ┆ [121.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, … 2.0] │
│ Fare        ┆ Float64 ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 16         ┆ 4.136  ┆ [113.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 18.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 3.0]        │
└─────────────┴─────────┴───────┴────────────┴───┴─────────────┴────────────┴────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x10e09af20>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTypeEnum.default: 1>, column_label='Min', column_align='center', column_width=None), ColInfo(var='25%', type=<ColInfoTypeEnum.default: 1>, column_label='25%', column_align='center', column_width=None), ColInfo(var='50%', type=<ColInfoTypeEnum.default: 1>, column_label='50%', column_align='center', column_width=None), ColInfo(var='75%', type=<ColInfoTypeEnum.default: 1>, column_label='75%', column_align='center', column_width=None), ColInfo(var='Max', type=<ColInfoTypeEnum.default: 1>, column_label='Max', column_align='center', column_width=None), ColInfo(var='IQR', type=<ColInfoTypeEnum.default: 1>, column_label='IQR', column_align='center', column_width=None), ColInfo(var='Pct_Missing', type=<ColInf

### 2.2 `include` - which columns to analyse

| Value | Meaning |
|-------|---------|
| `None` (default) | Numeric columns only |
| `'numeric'` | Same as None |
| `'all'` | Every column regardless of dtype |
| `'string'` | Only String / Utf8 columns |
| `'temporal'` | Only Date / Datetime columns |
| `['Float64', 'String']` | Specific dtype names |

In [7]:
# include='all' -> every column
ps.xray(df_titanic, include='all')

GT(_tbl_data=shape: (12, 16)
┌─────────────┬─────────┬───────┬────────────┬───┬─────────────┬────────────┬────────┬─────────────┐
│ Column      ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Pct_Missing ┆ N_Outliers ┆ skew   ┆ Distributio │
│ ---         ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---         ┆ ---        ┆ ---    ┆ n_Plot      │
│ str         ┆ str     ┆ i64   ┆ i64        ┆   ┆ f64         ┆ i64        ┆ f64    ┆ ---         │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ list[f64]   │
╞═════════════╪═════════╪═══════╪════════════╪═══╪═════════════╪════════════╪════════╪═════════════╡
│ PassengerId ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.0    ┆ [13.0,      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 13.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 13.0]       │
│ Survived    ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.647  ┆ [102.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, …      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 54.0]       │
│ Pclass      ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ -0.904 ┆ [30.0, 0.0, │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 96.0]     │
│ Name        ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ null   ┆ null        │
│ Sex         ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ null   ┆ null        │
│ …           ┆ …       ┆ …     ┆ …          ┆ … ┆ …           ┆ …          ┆ …      ┆ …           │
│ Parch       ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 35         ┆ 2.734  ┆ [121.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, … 2.0] │
│ Ticket      ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ null   ┆ null        │
│ Fare        ┆ Float64 ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 16         ┆ 4.136  ┆ [113.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 18.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 3.0]        │
│ Cabin       ┆ String  ┆ 31    ┆ 125        ┆ … ┆ 80.13       ┆ 0          ┆ null   ┆ null        │
│ Embarked    ┆ String  ┆ 155   ┆ 1          ┆ … ┆ 0.64        ┆ 0          ┆ null   ┆ null        │
└─────────────┴─────────┴───────┴────────────┴───┴─────────────┴────────────┴────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x11f7d7eb0>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTypeEnum.default: 1>, column_label='Min', column_align='center', column_width=None), ColInfo(var='25%', type=<ColInfoTypeEnum.default: 1>, column_label='25%', column_align='center', column_width=None), ColInfo(var='50%', type=<ColInfoTypeEnum.default: 1>, column_label='50%', column_align='center', column_width=None), ColInfo(var='75%', type=<ColInfoTypeEnum.default: 1>, column_label='75%', column_align='center', column_width=None), ColInfo(var='Max', type=<ColInfoTypeEnum.default: 1>, column_label='Ma

In [8]:
# include='numeric' -> same as default (None)
ps.xray(df_titanic, include='numeric')

GT(_tbl_data=shape: (7, 16)
┌─────────────┬─────────┬───────┬────────────┬───┬─────────────┬────────────┬────────┬─────────────┐
│ Column      ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Pct_Missing ┆ N_Outliers ┆ skew   ┆ Distributio │
│ ---         ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---         ┆ ---        ┆ ---    ┆ n_Plot      │
│ str         ┆ str     ┆ i64   ┆ i64        ┆   ┆ f64         ┆ i64        ┆ f64    ┆ ---         │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ list[f64]   │
╞═════════════╪═════════╪═══════╪════════════╪═══╪═════════════╪════════════╪════════╪═════════════╡
│ PassengerId ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.0    ┆ [13.0,      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 13.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 13.0]       │
│ Survived    ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.647  ┆ [102.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, …      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 54.0]       │
│ Pclass      ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ -0.904 ┆ [30.0, 0.0, │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 96.0]     │
│ Age         ┆ Float64 ┆ 126   ┆ 30         ┆ … ┆ 19.23       ┆ 4          ┆ 0.692  ┆ [9.0, 4.0,  │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 4.0]      │
│ SibSp       ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 12         ┆ 2.199  ┆ [98.0, 0.0, │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 2.0]      │
│ Parch       ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 35         ┆ 2.734  ┆ [121.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, … 2.0] │
│ Fare        ┆ Float64 ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 16         ┆ 4.136  ┆ [113.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 18.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 3.0]        │
└─────────────┴─────────┴───────┴────────────┴───┴─────────────┴────────────┴────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x11f838fa0>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTypeEnum.default: 1>, column_label='Min', column_align='center', column_width=None), ColInfo(var='25%', type=<ColInfoTypeEnum.default: 1>, column_label='25%', column_align='center', column_width=None), ColInfo(var='50%', type=<ColInfoTypeEnum.default: 1>, column_label='50%', column_align='center', column_width=None), ColInfo(var='75%', type=<ColInfoTypeEnum.default: 1>, column_label='75%', column_align='center', column_width=None), ColInfo(var='Max', type=<ColInfoTypeEnum.default: 1>, column_label='Max', column_align='center', column_width=None), ColInfo(var='IQR', type=<ColInfoTypeEnum.default: 1>, column_label='IQR', column_align='center', column_width=None), ColInfo(var='Pct_Missing', type=<ColInf

In [9]:
# include='string' -> only String/Utf8 columns
ps.xray(df_titanic, include='string')

GT(_tbl_data=shape: (5, 15)
┌──────────┬────────┬───────┬────────────┬───┬──────┬─────────────┬────────────┬───────────────────┐
│ Column   ┆ Dtype  ┆ Count ┆ null_count ┆ … ┆ IQR  ┆ Pct_Missing ┆ N_Outliers ┆ Distribution_Plot │
│ ---      ┆ ---    ┆ ---   ┆ ---        ┆   ┆ ---  ┆ ---         ┆ ---        ┆ ---               │
│ str      ┆ str    ┆ i64   ┆ i64        ┆   ┆ null ┆ f64         ┆ i64        ┆ null              │
╞══════════╪════════╪═══════╪════════════╪═══╪══════╪═════════════╪════════════╪═══════════════════╡
│ Name     ┆ String ┆ 156   ┆ 0          ┆ … ┆ null ┆ 0.0         ┆ 0          ┆ null              │
│ Sex      ┆ String ┆ 156   ┆ 0          ┆ … ┆ null ┆ 0.0         ┆ 0          ┆ null              │
│ Ticket   ┆ String ┆ 156   ┆ 0          ┆ … ┆ null ┆ 0.0         ┆ 0          ┆ null              │
│ Cabin    ┆ String ┆ 31    ┆ 125        ┆ … ┆ null ┆ 80.13       ┆ 0          ┆ null              │
│ Embarked ┆ String ┆ 155   ┆ 1          ┆ … ┆ null ┆ 0.64        ┆ 0          ┆ null              │
└──────────┴────────┴───────┴────────────┴───┴──────┴─────────────┴────────────┴───────────────────┘, _body=<great_tables._gt_data.Body object at 0x11f83acb0>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTypeEnum.default: 1>, column_label='Min', column_align='center', column_width=None), ColInfo(var='25%', type=<ColInfoTypeEnum.default: 1>, column_label='25%', column_align='center', column_width=None), ColInfo(var='50%', type=<ColInfoTypeEnum.default: 1>, column_label='50%', column_align='center', column_width=None), ColInfo(var='75%', type=<ColInfoTypeEnum.default: 1>, column_label='75%', column_align='center', column_width=None), ColInfo(var='Max', type=<ColInfoTypeEnum.default: 1>, column_label='Max', column_align='center', column_width=None), ColInfo(var='IQR', type=<ColInfoTypeEnum.default: 1>, column_label='IQR', column_align='center', column_width=None), ColInfo(var='Pct_Missing', type=<ColInfoTypeEnum.default: 1>, column_label='Pct_Missing', column_align='center', column_width=None), ColInfo(var='N_Outliers', type=<ColInfoTypeEnum.default: 1>, column_label='N_Outliers', column_align='center', column_width=None), ColInfo(var='Distribution_Plot', type=<ColInfoTypeEnum.default: 1>, column_label='Distribution_Plot', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11f7d5ed0>, _spanners=Spanners([SpannerInfo(spanner_id='Basic Statistics', spanner_level=0, spanner_label='Basic Statistics', spanner_units=None, spanner_pattern=None, vars=['Dtype', 'Count', 'null_count', 'Mean', 'std', 'Min', '25%', '50%', '75%', 'Max'], built=None), SpannerInfo(spanner_id='Key Metrics', spanner_level=0, spanner_label='Key Metrics', spanner_units=None, spanner_pattern=None, vars=['IQR', 'Pct_Missing', 'N_Outliers', 'Distribution_Plot'], built=None)]), _heading=Heading(title='🔬 DataFrame X-ray', subtitle='Dataset: 156 rows × 12 columns (15.3 KB in memory) - X-rayed in 3 ms', preheader=None), _stubhead=None, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x11f83bf70>, _formats=[<great_tables._gt_data.FormatInfo object at 0x107eb9a20>, <great_tables._gt_data.FormatInfo object at 0x11f83b370>, <great_tables._gt_data.Format

In [10]:
# include=['Float64', 'String'] -> specific dtype names
ps.xray(df_titanic, include=['Float64', 'String'])

GT(_tbl_data=shape: (7, 16)
┌──────────┬─────────┬───────┬────────────┬───┬─────────────┬────────────┬───────┬─────────────────┐
│ Column   ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Pct_Missing ┆ N_Outliers ┆ skew  ┆ Distribution_Pl │
│ ---      ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---         ┆ ---        ┆ ---   ┆ ot              │
│ str      ┆ str     ┆ i64   ┆ i64        ┆   ┆ f64         ┆ i64        ┆ f64   ┆ ---             │
│          ┆         ┆       ┆            ┆   ┆             ┆            ┆       ┆ list[f64]       │
╞══════════╪═════════╪═══════╪════════════╪═══╪═════════════╪════════════╪═══════╪═════════════════╡
│ Name     ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ null  ┆ null            │
│ Sex      ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ null  ┆ null            │
│ Age      ┆ Float64 ┆ 126   ┆ 30         ┆ … ┆ 19.23       ┆ 4          ┆ 0.692 ┆ [9.0, 4.0, …    │
│          ┆         ┆       ┆            ┆   ┆             ┆            ┆       ┆ 4.0]            │
│ Ticket   ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ null  ┆ null            │
│ Fare     ┆ Float64 ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 16         ┆ 4.136 ┆ [113.0, 18.0, … │
│          ┆         ┆       ┆            ┆   ┆             ┆            ┆       ┆ 3.0]            │
│ Cabin    ┆ String  ┆ 31    ┆ 125        ┆ … ┆ 80.13       ┆ 0          ┆ null  ┆ null            │
│ Embarked ┆ String  ┆ 155   ┆ 1          ┆ … ┆ 0.64        ┆ 0          ┆ null  ┆ null            │
└──────────┴─────────┴───────┴────────────┴───┴─────────────┴────────────┴───────┴─────────────────┘, _body=<great_tables._gt_data.Body object at 0x11f86c610>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTypeEnum.default: 1>, column_label='Min', column_align='center', column_width=None), ColInfo(var='25%', type=<ColInfoTypeEnum.default: 1>, column_label='25%', column_align='center', column_width=None), ColInfo(var='50%', type=<ColInfoTypeEnum.default: 1>, column_label='50%', column_align='center', column_width=None), ColInfo(var='75%', type=<ColInfoTypeEnum.default: 1>, column_label='75%', column_align='center', column_width=None), ColInfo(var='Max', type=<ColInfoTypeEnum.default: 1>, column_label='Max', column_align='center', column_width=None), ColInfo(var='IQR', type=<ColInfoTypeEnum.default: 1>, column_label='IQR', column_align='center', column_width=None), ColInfo(var='Pct_Missing', type=<ColInfoTypeEnum.default: 1>, column_label='Pct_Missing', column_align='center', column_width=None), ColInfo(var='N_Outliers', type=<ColInfoTypeEnum.default: 1>, column_label='N_Outliers', column_align='center', column_width=None), ColInfo(var='skew', type=<ColInfoTypeEnum.default: 1>, column_label='skew', column_align='center', column_width=None), ColInfo(var='Distribution_Plot', type=<ColInfoTypeEnum.default: 1>, column_label='Distribution_Plot', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11f7d5f60>, _spanners=Spanners([SpannerInfo(spanner_id='Basic Statistics', spanner_level=0, spanner_label='Basic Statistics', spanner_units=None, spanner_pattern=None, vars=['Dtype', 'Count', 'null_count', 'Mean', 'std', 'Min', '25%', '50%', '75%', 'Max'], built=None), S

### 2.3 `great_tables` - output format

In [11]:
# great_tables=True  -> formatted GT table (default, shown above)
# great_tables=False -> plain Polars DataFrame (useful for further processing)

result_df = ps.xray(df_titanic, great_tables=False)
print(f"Type: {type(result_df).__name__}")
result_df

Type: DataFrame


Column,Dtype,Count,null_count,Mean,std,Min,25%,50%,75%,Max,IQR,Pct_Missing,N_Outliers,skew,Distribution_Plot
str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,list[i64]
"""PassengerId""","""Int64""",156,0,78.5,45.177,1.0,40.0,79.0,117.0,156.0,77.0,0.0,0,0.0,"[13, 13, … 13]"
"""Survived""","""Int64""",156,0,0.346,0.477,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0,0.647,"[102, 0, … 54]"
"""Pclass""","""Int64""",156,0,2.423,0.795,1.0,2.0,3.0,3.0,3.0,1.0,0.0,0,-0.904,"[30, 0, … 96]"
"""Age""","""Float64""",126,30,28.142,14.614,0.83,19.0,26.0,35.0,71.0,16.0,19.23,4,0.692,"[9, 4, … 4]"
"""SibSp""","""Int64""",156,0,0.615,1.056,0.0,0.0,0.0,1.0,5.0,1.0,0.0,12,2.199,"[98, 0, … 2]"
"""Parch""","""Int64""",156,0,0.397,0.87,0.0,0.0,0.0,0.0,5.0,0.0,0.0,35,2.734,"[121, 0, … 2]"
"""Fare""","""Float64""",156,0,28.11,39.401,6.75,8.029,14.454,30.071,263.0,22.042,0.0,16,4.136,"[113, 18, … 3]"


### 2.4 `expanded` - full statistics mode

In [12]:
# expanded=False (default) -> essential metrics only:
#   Dtype, Count, null_count, Mean, std, Min, 25%, 50%, 75%, Max,
#   IQR, Pct_Missing, N_Outliers, skew, Distribution_Plot
#
# expanded=True -> adds:
#   MAD, Kurtosis, N_Unique, Uniqueness_Ratio, N_Duplicates, Pct_Duplicates,
#   N_Zero, Pct_Zero, Pct_Pos, Pct_Neg, Pct_Outliers,
#   Normality_Test, Uniformity_Test, Opt_Dtype, Shakiness_Score, Quality_Flag

ps.xray(df_diabetes, expanded=True)

GT(_tbl_data=shape: (9, 32)
┌────────────┬─────────┬───────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ Column     ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ N_Outliers ┆ Pct_Outlie ┆ Shakiness_ ┆ Quality_F │
│ ---        ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---        ┆ rs         ┆ Score      ┆ lag       │
│ str        ┆ str     ┆ i64   ┆ i64        ┆   ┆ i64        ┆ ---        ┆ ---        ┆ ---       │
│            ┆         ┆       ┆            ┆   ┆            ┆ f64        ┆ i64        ┆ str       │
╞════════════╪═════════╪═══════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ Pregnancie ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 4          ┆ 0.52       ┆ 1          ┆ ✓ OK      │
│ s          ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ Glucose    ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 5          ┆ 0.65       ┆ 1          ┆ ✓ OK      │
│ BloodPress ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 45         ┆ 5.86       ┆ 2          ┆ ⚠ SHAKY   │
│ ure        ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ SkinThickn ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 1          ┆ 0.13       ┆ 1          ┆ ✓ OK      │
│ ess        ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ Insulin    ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 35         ┆ 4.56       ┆ 3          ┆ ⚠ SHAKY   │
│ BMI        ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 19         ┆ 2.47       ┆ 1          ┆ ✓ OK      │
│ DiabetesPe ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 29         ┆ 3.78       ┆ 1          ┆ ✓ OK      │
│ digreeFunc ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ tion       ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ Age        ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 9          ┆ 1.17       ┆ 1          ┆ ✓ OK      │
│ Outcome    ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0          ┆ 0.0        ┆ 2          ┆ ⚠ SHAKY   │
└────────────┴─────────┴───────┴────────────┴───┴────────────┴────────────┴────────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x12f0a6350>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTypeEnum.default: 1>, column_label='Min', column_align='center', column_width=None), ColInfo(var='Max', type=<ColInfoTypeEnum.default: 1>, column_label='Max', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Pct_Missing', type=<ColInfoTypeEnum.default: 1>, column_label='Pct_Missing', column_align='center', column_width=None), ColInfo(var='N_Unique', type=<ColInfoTypeEnum.default: 1>, column_label='N_Unique', column_align='center', column_width=None), ColInfo(var='Uniqueness_Ratio', type=<ColInfoTypeEnum.default: 1>, column_label='Uniqueness_Ratio', column_align='center', column_width=None), ColInfo(var='N_Duplicates', type=<ColInfoTypeEnum.default: 1>, column_label='N_Duplicates', column_align='center', column_width=None), ColInfo(var='Pct_Duplicates', type=<ColInfoTypeEnum.default: 1>, column_label='Pct_Duplicates', column_align='center', column_width=None), ColInfo(var='N_Zero', type=<ColInfoTypeEnum.default: 1>, column_label='N_Zero', column_align='center', column_width=None), ColInfo(va

In [13]:
# expanded=True with include='all' on mixed-type data
ps.xray(df_titanic, expanded=True, include='all')

GT(_tbl_data=shape: (12, 32)
┌────────────┬─────────┬───────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ Column     ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ N_Outliers ┆ Pct_Outlie ┆ Shakiness_ ┆ Quality_F │
│ ---        ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---        ┆ rs         ┆ Score      ┆ lag       │
│ str        ┆ str     ┆ i64   ┆ i64        ┆   ┆ i64        ┆ ---        ┆ ---        ┆ ---       │
│            ┆         ┆       ┆            ┆   ┆            ┆ f64        ┆ i64        ┆ str       │
╞════════════╪═════════╪═══════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ PassengerI ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0          ┆ 0.0        ┆ 2          ┆ ⚠ SHAKY   │
│ d          ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ Survived   ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0          ┆ 0.0        ┆ 1          ┆ ✓ OK      │
│ Pclass     ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0          ┆ 0.0        ┆ 1          ┆ ✓ OK      │
│ Name       ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 0          ┆ 0.0        ┆ 1          ┆ ✓ OK      │
│ Sex        ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 0          ┆ 0.0        ┆ 0          ┆ ✓ OK      │
│ …          ┆ …       ┆ …     ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …         │
│ Parch      ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 35         ┆ 22.44      ┆ 4          ┆ ⚠ SHAKY   │
│ Ticket     ┆ String  ┆ 156   ┆ 0          ┆ … ┆ 0          ┆ 0.0        ┆ 0          ┆ ✓ OK      │
│ Fare       ┆ Float64 ┆ 156   ┆ 0          ┆ … ┆ 16         ┆ 10.26      ┆ 4          ┆ ⚠ SHAKY   │
│ Cabin      ┆ String  ┆ 31    ┆ 125        ┆ … ┆ 0          ┆ 0.0        ┆ 1          ┆ ✓ OK      │
│ Embarked   ┆ String  ┆ 155   ┆ 1          ┆ … ┆ 0          ┆ 0.0        ┆ 0          ┆ ✓ OK      │
└────────────┴─────────┴───────┴────────────┴───┴────────────┴────────────┴────────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x12f0a4130>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTypeEnum.default: 1>, column_label='Min', column_align='center', column_width=None), ColInfo(var='Max', type=<ColInfoTypeEnum.default: 1>, column_label='Max', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Pct_Missing', type=<ColInfoTypeEnum.default: 1>, column_label='Pct_Missing', column_align='center', column_width=None), ColInfo(var='N_Unique', type=<ColInfoTypeEnum.default: 1>, column_label='N_Unique', column_align='center', column_width=None), ColInfo(var='Uniqueness_Ratio', type=<ColInfoTypeEnum.default: 1>, column_label='Uniqueness_Ratio', column_align='center', column_width=None), ColInfo(var='N_Duplicates', type=<ColInfoTypeEnum.default: 1>, column_label='N_Duplicates', column_align='center', column_width=None), ColInfo(var='Pct_Duplicates', type=<ColInfoTypeEnum.default: 1>, column_label='Pct_Duplicates', column_align='center', column_width=None), ColInfo(var='N_Zero', type=<ColInfoTypeEnum.default: 1>, column_label='N_Zero', column_align='center', column_width=None), ColInfo(var='Pct_Zero', type=<ColInfoTypeEnum.default: 1>, column_label='Pct_Zero', column_align='center', column_width=None), ColInfo(var='Pct_Pos', type=<ColInfoTypeEnum.default: 1>, column_label='Pct_Pos', co

### 2.5 `title` - custom table title

In [14]:
# title=None (default) -> 'DataFrame X-ray' or 'Expanded Statistics'
# title='My Title'     -> custom title (only affects GT output)

ps.xray(df_titanic, title="Titanic Passenger Analysis")

GT(_tbl_data=shape: (7, 16)
┌─────────────┬─────────┬───────┬────────────┬───┬─────────────┬────────────┬────────┬─────────────┐
│ Column      ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Pct_Missing ┆ N_Outliers ┆ skew   ┆ Distributio │
│ ---         ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---         ┆ ---        ┆ ---    ┆ n_Plot      │
│ str         ┆ str     ┆ i64   ┆ i64        ┆   ┆ f64         ┆ i64        ┆ f64    ┆ ---         │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ list[f64]   │
╞═════════════╪═════════╪═══════╪════════════╪═══╪═════════════╪════════════╪════════╪═════════════╡
│ PassengerId ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.0    ┆ [13.0,      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 13.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 13.0]       │
│ Survived    ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.647  ┆ [102.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, …      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 54.0]       │
│ Pclass      ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ -0.904 ┆ [30.0, 0.0, │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 96.0]     │
│ Age         ┆ Float64 ┆ 126   ┆ 30         ┆ … ┆ 19.23       ┆ 4          ┆ 0.692  ┆ [9.0, 4.0,  │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 4.0]      │
│ SibSp       ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 12         ┆ 2.199  ┆ [98.0, 0.0, │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 2.0]      │
│ Parch       ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 35         ┆ 2.734  ┆ [121.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, … 2.0] │
│ Fare        ┆ Float64 ┆ 156   ┆ 0          ┆ … ┆ 0.0         ┆ 16         ┆ 4.136  ┆ [113.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 18.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 3.0]        │
└─────────────┴─────────┴───────┴────────────┴───┴─────────────┴────────────┴────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x12f036e30>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTypeEnum.default: 1>, column_label='Min', column_align='center', column_width=None), ColInfo(var='25%', type=<ColInfoTypeEnum.default: 1>, column_label='25%', column_align='center', column_width=None), ColInfo(var='50%', type=<ColInfoTypeEnum.default: 1>, column_label='50%', column_align='center', column_width=None), ColInfo(var='75%', type=<ColInfoTypeEnum.default: 1>, column_label='75%', column_align='center', column_width=None), ColInfo(var='Max', type=<ColInfoTypeEnum.default: 1>, column_label='Max', column_align='center', column_width=None), ColInfo(var='IQR', type=<ColInfoTypeEnum.default: 1>, column_label='IQR', column_align='center', column_width=None), ColInfo(var='Pct_Missing', type=<ColInf

### 2.6 `percentiles` - custom quantiles

In [15]:
# percentiles=None (default) -> [0.25, 0.5, 0.75]  (shows 25%, 50%, 75% columns)
# percentiles=[0.1, 0.5, 0.9] -> shows 10%, 50%, 90% columns
# Values must be between 0 and 1; list must be non-empty

# Extended percentiles
ps.xray(df_diabetes, percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

GT(_tbl_data=shape: (9, 16)
┌─────────────┬─────────┬───────┬────────────┬───┬─────────────┬────────────┬────────┬─────────────┐
│ Column      ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Pct_Missing ┆ N_Outliers ┆ skew   ┆ Distributio │
│ ---         ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---         ┆ ---        ┆ ---    ┆ n_Plot      │
│ str         ┆ str     ┆ i64   ┆ i64        ┆   ┆ f64         ┆ i64        ┆ f64    ┆ ---         │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ list[f64]   │
╞═════════════╪═════════╪═══════╪════════════╪═══╪═════════════╪════════════╪════════╪═════════════╡
│ Pregnancies ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 4          ┆ 0.9    ┆ [246.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 103.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Glucose     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 5          ┆ 0.173  ┆ [5.0, 0.0,  │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 35.0]     │
│ BloodPressu ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 45         ┆ -1.84  ┆ [35.0, 0.0, │
│ re          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 2.0]      │
│ SkinThickne ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 1          ┆ 0.109  ┆ [231.0,     │
│ ss          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 55.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Insulin     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 35         ┆ 2.268  ┆ [456.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 148.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ BMI         ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 19         ┆ -0.428 ┆ [11.0, 0.0, │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 1.0]      │
│ DiabetesPed ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 29         ┆ 1.916  ┆ [266.0,     │
│ igreeFuncti ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 205.0, …    │
│ on          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 3.0]        │
│ Age         ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 9          ┆ 1.127  ┆ [267.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 150.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Outcome     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.634  ┆ [500.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, …      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 268.0]      │
└─────────────┴─────────┴───────┴────────────┴───┴─────────────┴────────────┴────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x12f1b42e0>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTy

In [16]:
# Extreme quantiles (as DataFrame for easier reading)
ps.xray(df_diabetes, percentiles=[0.01, 0.05, 0.5, 0.95, 0.99], great_tables=False)

Column,Dtype,Count,null_count,Mean,std,Min,50%,Max,Pct_Missing,N_Outliers,skew,Distribution_Plot
str,str,i64,i64,f64,f64,f64,f64,f64,f64,i64,f64,list[i64]
"""Pregnancies""","""Int64""",768,0,3.845,3.37,0.0,3.0,17.0,0.0,4,0.9,"[246, 103, … 1]"
"""Glucose""","""Int64""",768,0,120.895,31.973,0.0,117.0,199.0,0.0,5,0.173,"[5, 0, … 35]"
"""BloodPressure""","""Int64""",768,0,69.105,19.356,0.0,72.0,122.0,0.0,45,-1.84,"[35, 0, … 2]"
"""SkinThickness""","""Int64""",768,0,20.536,15.952,0.0,23.0,99.0,0.0,1,0.109,"[231, 55, … 1]"
"""Insulin""","""Int64""",768,0,79.799,115.244,0.0,32.0,846.0,0.0,35,2.268,"[456, 148, … 1]"
"""BMI""","""Float64""",768,0,31.993,7.884,0.0,32.0,67.1,0.0,19,-0.428,"[11, 0, … 1]"
"""DiabetesPedigreeFunction""","""Float64""",768,0,0.472,0.331,0.078,0.374,2.42,0.0,29,1.916,"[266, 205, … 3]"
"""Age""","""Int64""",768,0,33.241,11.76,21.0,29.0,81.0,0.0,9,1.127,"[267, 150, … 1]"
"""Outcome""","""Int64""",768,0,0.349,0.477,0.0,0.0,1.0,0.0,0,0.634,"[500, 0, … 268]"


### 2.7 `outlier_method` & `outlier_bounds`

| `outlier_method` | How it works | `outlier_bounds` required? |
|------------------|--------------|---------------------------|
| `'iqr'` (default) | Values below Q1-1.5*IQR or above Q3+1.5*IQR | No |
| `'zscore'` | Values where \|z-score\| > 3 | No |
| `'percentile'` | Values outside `[lower_pct, upper_pct]` bounds | **Yes** - e.g. `[0.05, 0.95]` |

In [17]:
# outlier_method='iqr' (default)
iqr_result = ps.xray(df_diabetes, outlier_method='iqr', great_tables=False)
print("IQR method:")
iqr_result.select('Column', 'N_Outliers', 'skew')

IQR method:


Column,N_Outliers,skew
str,i64,f64
"""Pregnancies""",4,0.9
"""Glucose""",5,0.173
"""BloodPressure""",45,-1.84
"""SkinThickness""",1,0.109
"""Insulin""",35,2.268
"""BMI""",19,-0.428
"""DiabetesPedigreeFunction""",29,1.916
"""Age""",9,1.127
"""Outcome""",0,0.634


In [18]:
# outlier_method='zscore'
zs_result = ps.xray(df_diabetes, outlier_method='zscore', great_tables=False)
print("Z-score method:")
zs_result.select('Column', 'N_Outliers', 'skew')

Z-score method:


Column,N_Outliers,skew
str,i64,f64
"""Pregnancies""",4,0.9
"""Glucose""",5,0.173
"""BloodPressure""",35,-1.84
"""SkinThickness""",1,0.109
"""Insulin""",18,2.268
"""BMI""",14,-0.428
"""DiabetesPedigreeFunction""",11,1.916
"""Age""",5,1.127
"""Outcome""",0,0.634


In [19]:
# outlier_method='percentile', outlier_bounds=[lower, upper]
pct_result = ps.xray(
    df_diabetes,
    outlier_method='percentile',
    outlier_bounds=[0.05, 0.95],   # values: [lower_percentile, upper_percentile]
    great_tables=False
)
print("Percentile method (5th-95th):")
pct_result.select('Column', 'N_Outliers', 'skew')

Percentile method (5th-95th):


Column,N_Outliers,skew
str,i64,f64
"""Pregnancies""",34,0.9
"""Glucose""",74,0.173
"""BloodPressure""",76,-1.84
"""SkinThickness""",37,0.109
"""Insulin""",38,2.268
"""BMI""",75,-0.428
"""DiabetesPedigreeFunction""",75,1.916
"""Age""",35,1.127
"""Outcome""",0,0.634


### 2.8 `corr_target` - correlation with a target column

In [20]:
# corr_target=None (default) -> no correlation column
# corr_target='Outcome'      -> adds Correlation and Correlation_Plot columns
#   The target column itself shows '-' for its correlation value
#   Non-numeric columns show None

ps.xray(df_diabetes, corr_target='Outcome')

GT(_tbl_data=shape: (9, 18)
┌─────────────┬─────────┬───────┬────────────┬───┬────────┬─────────────┬─────────────┬────────────┐
│ Column      ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ skew   ┆ Distributio ┆ Correlation ┆ Correlatio │
│ ---         ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---    ┆ n_Plot      ┆ ---         ┆ n_Plot     │
│ str         ┆ str     ┆ i64   ┆ i64        ┆   ┆ f64    ┆ ---         ┆ str         ┆ ---        │
│             ┆         ┆       ┆            ┆   ┆        ┆ list[f64]   ┆             ┆ f64        │
╞═════════════╪═════════╪═══════╪════════════╪═══╪════════╪═════════════╪═════════════╪════════════╡
│ Pregnancies ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.9    ┆ [246.0,     ┆ 0.222       ┆ 0.222      │
│             ┆         ┆       ┆            ┆   ┆        ┆ 103.0, …    ┆             ┆            │
│             ┆         ┆       ┆            ┆   ┆        ┆ 1.0]        ┆             ┆            │
│ Glucose     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.173  ┆ [5.0, 0.0,  ┆ 0.467       ┆ 0.467      │
│             ┆         ┆       ┆            ┆   ┆        ┆ … 35.0]     ┆             ┆            │
│ BloodPressu ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ -1.84  ┆ [35.0, 0.0, ┆ 0.065       ┆ 0.065      │
│ re          ┆         ┆       ┆            ┆   ┆        ┆ … 2.0]      ┆             ┆            │
│ SkinThickne ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.109  ┆ [231.0,     ┆ 0.075       ┆ 0.075      │
│ ss          ┆         ┆       ┆            ┆   ┆        ┆ 55.0, …     ┆             ┆            │
│             ┆         ┆       ┆            ┆   ┆        ┆ 1.0]        ┆             ┆            │
│ Insulin     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 2.268  ┆ [456.0,     ┆ 0.131       ┆ 0.131      │
│             ┆         ┆       ┆            ┆   ┆        ┆ 148.0, …    ┆             ┆            │
│             ┆         ┆       ┆            ┆   ┆        ┆ 1.0]        ┆             ┆            │
│ BMI         ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ -0.428 ┆ [11.0, 0.0, ┆ 0.293       ┆ 0.293      │
│             ┆         ┆       ┆            ┆   ┆        ┆ … 1.0]      ┆             ┆            │
│ DiabetesPed ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 1.916  ┆ [266.0,     ┆ 0.174       ┆ 0.174      │
│ igreeFuncti ┆         ┆       ┆            ┆   ┆        ┆ 205.0, …    ┆             ┆            │
│ on          ┆         ┆       ┆            ┆   ┆        ┆ 3.0]        ┆             ┆            │
│ Age         ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 1.127  ┆ [267.0,     ┆ 0.238       ┆ 0.238      │
│             ┆         ┆       ┆            ┆   ┆        ┆ 150.0, …    ┆             ┆            │
│             ┆         ┆       ┆            ┆   ┆        ┆ 1.0]        ┆             ┆            │
│ Outcome     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.634  ┆ [500.0,     ┆ -           ┆ null       │
│             ┆         ┆       ┆            ┆   ┆        ┆ 0.0, …      ┆             ┆            │
│             ┆         ┆       ┆            ┆   ┆        ┆ 268.0]      ┆             ┆            │
└─────────────┴─────────┴───────┴────────────┴───┴────────┴─────────────┴─────────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x12f1b5870>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTy

In [21]:
# corr_target works in expanded mode too
ps.xray(df_diabetes, corr_target='Outcome', expanded=True)

GT(_tbl_data=shape: (9, 34)
┌────────────┬─────────┬───────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ Column     ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Correlatio ┆ Correlatio ┆ Shakiness_ ┆ Quality_F │
│ ---        ┆ ---     ┆ ---   ┆ ---        ┆   ┆ n          ┆ n_Plot     ┆ Score      ┆ lag       │
│ str        ┆ str     ┆ i64   ┆ i64        ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│            ┆         ┆       ┆            ┆   ┆ str        ┆ f64        ┆ i64        ┆ str       │
╞════════════╪═════════╪═══════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ Pregnancie ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.222      ┆ 0.222      ┆ 1          ┆ ✓ OK      │
│ s          ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ Glucose    ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.467      ┆ 0.467      ┆ 1          ┆ ✓ OK      │
│ BloodPress ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.065      ┆ 0.065      ┆ 2          ┆ ⚠ SHAKY   │
│ ure        ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ SkinThickn ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.075      ┆ 0.075      ┆ 1          ┆ ✓ OK      │
│ ess        ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ Insulin    ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.131      ┆ 0.131      ┆ 3          ┆ ⚠ SHAKY   │
│ BMI        ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.293      ┆ 0.293      ┆ 1          ┆ ✓ OK      │
│ DiabetesPe ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.174      ┆ 0.174      ┆ 1          ┆ ✓ OK      │
│ digreeFunc ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ tion       ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆           │
│ Age        ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.238      ┆ 0.238      ┆ 1          ┆ ✓ OK      │
│ Outcome    ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ -          ┆ null       ┆ 2          ┆ ⚠ SHAKY   │
└────────────┴─────────┴───────┴────────────┴───┴────────────┴────────────┴────────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x12f1d94b0>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTypeEnum.default: 1>, column_label='Min', column_align='center', column_width=None), ColInfo(var='Max', type=<ColInfoTypeEnum.default: 1>, column_label='Max', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Pct_Missing', type=<ColInfoTypeEnum.default: 1>, column_label='Pct_Missing', column_align='center', column_width=None), ColInfo(var='N_Unique', type=<ColInfoTypeEnum.default: 1>, column_label='N_Unique', column_align='center', column_width=None), ColInfo(var='Uniqueness_Ratio', type=<ColInfoTypeEnum.default: 1>, column_label='Uniqueness_Ratio', column_align='center', column_width=None), ColInfo(var='N_Duplicates', type=<ColInfoTypeEnum.default: 1>, column_label='N_Duplicates', column_align='center', column_width=None), ColInfo(var='Pct_Duplicates', type=<ColInfoTypeEnum.default: 1>, column_label='Pct_Duplicates', column_align='center', column_width=None), ColInfo(var='N_Zero', type=<ColInfoTypeEnum.default: 1>, column_label='N_Zero', column_align='center', column_width=None), ColInfo(va

### 2.9 `normality_test` - statistical normality test (expanded only)

| Value | Test | Best for | Notes |
|-------|------|----------|-------|
| `'shapiro'` (default) | Shapiro-Wilk | Small/medium samples (n < 5000) | Most powerful for small n; reports "N/A" for n > 5000 |
| `'anderson'` | Anderson-Darling | Sensitive to tails | Uses critical values, no exact p-value |
| `'ks'` | Kolmogorov-Smirnov | General purpose | Compares against theoretical normal |

> **Interpreting results:** Most real-world data is **not** perfectly normal. Seeing "NON-NORMAL"
> everywhere is expected, especially with enough data points. The test is useful for flagging
> *how far* from normal the data is, not for expecting normality.

In [22]:
# normality_test='shapiro' (default)
ps.xray(df_diabetes, expanded=True, normality_test='shapiro', great_tables=False).select(
    'Column', 'Normality_Test'
)

Column,Normality_Test
str,str
"""Pregnancies""","""NON-NORMAL (Shapiro-Wilk, p=0.…"
"""Glucose""","""NON-NORMAL (Shapiro-Wilk, p=0.…"
"""BloodPressure""","""NON-NORMAL (Shapiro-Wilk, p=0.…"
"""SkinThickness""","""NON-NORMAL (Shapiro-Wilk, p=0.…"
"""Insulin""","""NON-NORMAL (Shapiro-Wilk, p=0.…"
"""BMI""","""NON-NORMAL (Shapiro-Wilk, p=0.…"
"""DiabetesPedigreeFunction""","""NON-NORMAL (Shapiro-Wilk, p=0.…"
"""Age""","""NON-NORMAL (Shapiro-Wilk, p=0.…"
"""Outcome""","""NON-NORMAL (Shapiro-Wilk, p=0.…"


In [23]:
# normality_test='anderson'
ps.xray(df_diabetes, expanded=True, normality_test='anderson', great_tables=False).select(
    'Column', 'Normality_Test'
)

Column,Normality_Test
str,str
"""Pregnancies""","""NON-NORMAL (Anderson-Darling)"""
"""Glucose""","""NON-NORMAL (Anderson-Darling)"""
"""BloodPressure""","""NON-NORMAL (Anderson-Darling)"""
"""SkinThickness""","""NON-NORMAL (Anderson-Darling)"""
"""Insulin""","""NON-NORMAL (Anderson-Darling)"""
"""BMI""","""NON-NORMAL (Anderson-Darling)"""
"""DiabetesPedigreeFunction""","""NON-NORMAL (Anderson-Darling)"""
"""Age""","""NON-NORMAL (Anderson-Darling)"""
"""Outcome""","""NON-NORMAL (Anderson-Darling)"""


In [24]:
# normality_test='ks'
ps.xray(df_diabetes, expanded=True, normality_test='ks', great_tables=False).select(
    'Column', 'Normality_Test'
)

Column,Normality_Test
str,str
"""Pregnancies""","""NON-NORMAL (Kolmogorov-Smirnov…"
"""Glucose""","""NON-NORMAL (Kolmogorov-Smirnov…"
"""BloodPressure""","""NON-NORMAL (Kolmogorov-Smirnov…"
"""SkinThickness""","""NON-NORMAL (Kolmogorov-Smirnov…"
"""Insulin""","""NON-NORMAL (Kolmogorov-Smirnov…"
"""BMI""","""NORMAL (Kolmogorov-Smirnov, p=…"
"""DiabetesPedigreeFunction""","""NON-NORMAL (Kolmogorov-Smirnov…"
"""Age""","""NON-NORMAL (Kolmogorov-Smirnov…"
"""Outcome""","""NON-NORMAL (Kolmogorov-Smirnov…"


### 2.10 `uniformity_test` - statistical uniformity test (expanded only)

| Value | Test | Description |
|-------|------|-------------|
| `'ks'` (default) | Kolmogorov-Smirnov | Tests if data follows a uniform distribution |
| `'chi2'` | Chi-square goodness of fit | Bins data and checks equal frequency |

In [25]:
# uniformity_test='ks' (default)
ps.xray(df_diabetes, expanded=True, uniformity_test='ks', great_tables=False).select(
    'Column', 'Uniformity_Test'
)

Column,Uniformity_Test
str,str
"""Pregnancies""","""NON-UNIFORM (KS, p=0.000)"""
"""Glucose""","""NON-UNIFORM (KS, p=0.000)"""
"""BloodPressure""","""NON-UNIFORM (KS, p=0.000)"""
"""SkinThickness""","""NON-UNIFORM (KS, p=0.000)"""
"""Insulin""","""NON-UNIFORM (KS, p=0.000)"""
"""BMI""","""NON-UNIFORM (KS, p=0.000)"""
"""DiabetesPedigreeFunction""","""NON-UNIFORM (KS, p=0.000)"""
"""Age""","""NON-UNIFORM (KS, p=0.000)"""
"""Outcome""","""NON-UNIFORM (KS, p=0.000)"""


In [26]:
# uniformity_test='chi2'
ps.xray(df_diabetes, expanded=True, uniformity_test='chi2', great_tables=False).select(
    'Column', 'Uniformity_Test'
)

Column,Uniformity_Test
str,str
"""Pregnancies""","""NON-UNIFORM (Chi-square, p=0.0…"
"""Glucose""","""NON-UNIFORM (Chi-square, p=0.0…"
"""BloodPressure""","""NON-UNIFORM (Chi-square, p=0.0…"
"""SkinThickness""","""NON-UNIFORM (Chi-square, p=0.0…"
"""Insulin""","""NON-UNIFORM (Chi-square, p=0.0…"
"""BMI""","""NON-UNIFORM (Chi-square, p=0.0…"
"""DiabetesPedigreeFunction""","""NON-UNIFORM (Chi-square, p=0.0…"
"""Age""","""NON-UNIFORM (Chi-square, p=0.0…"
"""Outcome""","""NON-UNIFORM (Chi-square, p=0.0…"


### 2.11 Quality threshold parameters

These control when a column gets flagged in the `Shakiness_Score` / `Quality_Flag`.
Each threshold violation adds +1 to the shakiness score.

| Parameter | Default | Flags when |
|-----------|---------|------------|
| `missing_threshold` | `0.3` | % missing > threshold * 100 |
| `constant_threshold` | `0.99` | Uniqueness ratio < (1 - threshold) |
| `skew_threshold` | `2.0` | \|skew\| > threshold |
| `kurtosis_threshold` | `7.0` | \|kurtosis\| > threshold |
| `outlier_threshold` | `0.05` | % outliers > threshold * 100 |
| `shakiness_threshold` | `2` | Score >= this -> column is "SHAKY" |

In [27]:
# Build a DataFrame with known quality issues
np.random.seed(42)
df_quality = pl.DataFrame({
    'clean_col':    np.random.normal(50, 10, 200),
    'high_missing': [None]*140 + list(np.random.normal(0, 1, 60)),
    'constant':     [42.0] * 200,
    'skewed':       np.random.exponential(2, 200),
    'outlier_heavy': np.concatenate([np.random.normal(0, 1, 180),
                                     np.array([100, -100]*10)]),
})

# STRICT thresholds -> more columns flagged as SHAKY
ps.xray(
    df_quality,
    expanded=True,
    missing_threshold=0.1,       # flag if >10% missing (strict)
    constant_threshold=0.95,     # flag quasi-constant (strict)
    skew_threshold=1.0,          # flag |skew| > 1 (strict)
    kurtosis_threshold=3.0,      # flag |kurtosis| > 3 (strict)
    outlier_threshold=0.01,      # flag if >1% outliers (strict)
    shakiness_threshold=1,       # any single flag = SHAKY
    great_tables=False
).select('Column', 'Pct_Missing', 'skew', 'Kurtosis', 'Pct_Outliers',
         'Shakiness_Score', 'Quality_Flag')

/Users/anders/miniconda3/envs/polars_env/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:586: UserWarning: scipy.stats.shapiro: Input data has range zero. The results may not be accurate.
  res = hypotest_fun_out(*samples, **kwds)


Column,Pct_Missing,skew,Kurtosis,Pct_Outliers,Shakiness_Score,Quality_Flag
str,f64,f64,f64,f64,i64,str
"""clean_col""",0.0,0.132,-0.032,1.5,2,"""⚠ SHAKY"""
"""high_missing""",70.0,0.491,0.765,1.67,2,"""⚠ SHAKY"""
"""constant""",0.0,NaN,NaN,0.0,1,"""⚠ SHAKY"""
"""skewed""",0.0,1.96,5.526,4.0,5,"""⚠ SHAKY"""
"""outlier_heavy""",0.0,0.006,6.882,10.5,3,"""⚠ SHAKY"""


In [28]:
# LENIENT thresholds -> fewer flags
ps.xray(
    df_quality,
    expanded=True,
    missing_threshold=0.8,       # flag only if >80% missing
    skew_threshold=5.0,          # only very extreme skew
    kurtosis_threshold=20.0,     # only very extreme kurtosis
    outlier_threshold=0.2,       # flag only if >20% outliers
    shakiness_threshold=4,       # need 4+ issues for SHAKY
    great_tables=False
).select('Column', 'Shakiness_Score', 'Quality_Flag')

Column,Shakiness_Score,Quality_Flag
str,i64,str
"""clean_col""",1,"""✓ OK"""
"""high_missing""",0,"""✓ OK"""
"""constant""",1,"""✓ OK"""
"""skewed""",2,"""✓ OK"""
"""outlier_heavy""",1,"""✓ OK"""


### 2.12 `model_usability` - ML readiness assessment

When `model_usability=True`, three extra columns are added:

- **Usability_Flags**: Comma-separated flag codes indicating issues
- **Usability_Score**: 0-100 score (higher = more usable for ML)
- **Recommendation**: Actionable text ("Good for modeling", "Drop - constant value", etc.)

#### Flag code reference

| Flag | Meaning | Weight | Triggered when |
|------|---------|--------|----------------|
| `HM` | High Missing | 5.0 | > 90% null |
| `MM` | Moderate Missing | 3.0 | > 50% null |
| `ID` | Likely ID column | 4.0 | > 95% unique or ID-like name + high cardinality |
| `BN` | Binary column | 1.0 | Exactly 2 unique values |
| `CV` | Constant Value | 5.0 | Only 1 unique value |
| `EO` | Extreme Outliers | 2.5 | > 10% outliers |
| `ES` | Extreme Skew | 2.0 | \|skew\| > 3 |
| `EK` | Extreme Kurtosis | 2.0 | \|kurtosis\| > 7 |
| `NN` | Non-Normal | 1.5 | Failed normality test |
| `ZH` | Zero-Heavy | 2.5 | > 80% zeros |
| `UC` | Unreliable Correlation | 1.0 | Non-normal + corr_target set |

#### Recommendation meanings

| Score range | Recommendation | Action |
|-------------|---------------|--------|
| 90-100 | "Good for modeling" | Use as-is |
| 70-89 | "Minor issues - review" | Check the flagged issues, likely still usable |
| 50-69 | "Review before using" | Investigate and possibly transform |
| 30-49 | "Use with caution" | Consider transformations or removal |
| 0-29 | "Use with extreme caution" | Likely needs significant preprocessing |
| - | "Drop - constant value" | Column has zero variance |
| - | "Drop - too many missing values" | > 90% null |
| - | "Drop - likely an ID column" | Not a feature |

In [29]:
# model_usability=False (default) -> no usability columns
# model_usability=True            -> adds Usability_Flags, Usability_Score, Recommendation
#   Works best with expanded=True (for kurtosis, normality checks)
#   Combine with corr_target for UC (Unreliable Correlation) flag

ps.xray(
    df_titanic,
    include='all',
    expanded=True,
    model_usability=True,
    corr_target='Survived',
    great_tables=False
).select('Column', 'Dtype', 'Usability_Flags', 'Usability_Score', 'Recommendation')

Column,Dtype,Usability_Flags,Usability_Score,Recommendation
str,str,str,f64,str
"""PassengerId""","""Int64""","""ID,NN,UC""",77.966102,"""Drop - likely an ID column"""
"""Survived""","""Int64""","""BN,NN,UC""",88.135593,"""Minor issues - review"""
"""Pclass""","""Int64""","""NN,UC""",91.525424,"""Good for modeling"""
"""Name""","""String""","""ID""",86.440678,"""Drop - likely an ID column"""
"""Sex""","""String""","""BN""",96.610169,"""Good for modeling"""
…,…,…,…,…
"""Parch""","""Int64""","""EK,EO,NN,UC""",76.271186,"""Minor issues - review"""
"""Ticket""","""String""","""-""",100.0,"""Good for modeling"""
"""Fare""","""Float64""","""EK,EO,ES,NN,UC""",69.491525,"""Review before using"""


In [30]:
# Full GT-rendered model usability view
ps.xray(
    df_titanic,
    include='all',
    expanded=True,
    model_usability=True,
    corr_target='Survived',
    title="Titanic - Model Usability Assessment"
)

GT(_tbl_data=shape: (12, 37)
┌────────────┬─────────┬───────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ Column     ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Quality_Fl ┆ Usability_ ┆ Usability_ ┆ Recommend │
│ ---        ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ag         ┆ Flags      ┆ Score      ┆ ation     │
│ str        ┆ str     ┆ i64   ┆ i64        ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│            ┆         ┆       ┆            ┆   ┆ str        ┆ str        ┆ f64        ┆ str       │
╞════════════╪═════════╪═══════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ PassengerI ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ ⚠ SHAKY    ┆ ID,NN,UC   ┆ 77.966102  ┆ Drop -    │
│ d          ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ likely an │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ ID column │
│ Survived   ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ ✓ OK       ┆ BN,NN,UC   ┆ 88.135593  ┆ Minor     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ issues -  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ review    │
│ Pclass     ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ ✓ OK       ┆ NN,UC      ┆ 91.525424  ┆ Good for  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ Name       ┆ String  ┆ 156   ┆ 0          ┆ … ┆ ✓ OK       ┆ ID         ┆ 86.440678  ┆ Drop -    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ likely an │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ ID column │
│ Sex        ┆ String  ┆ 156   ┆ 0          ┆ … ┆ ✓ OK       ┆ BN         ┆ 96.610169  ┆ Good for  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ …          ┆ …       ┆ …     ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …         │
│ Parch      ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ ⚠ SHAKY    ┆ EK,EO,NN,U ┆ 76.271186  ┆ Minor     │
│            ┆         ┆       ┆            ┆   ┆            ┆ C          ┆            ┆ issues -  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ review    │
│ Ticket     ┆ String  ┆ 156   ┆ 0          ┆ … ┆ ✓ OK       ┆ -          ┆ 100.0      ┆ Good for  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ Fare       ┆ Float64 ┆ 156   ┆ 0          ┆ … ┆ ⚠ SHAKY    ┆ EK,EO,ES,N ┆ 69.491525  ┆ Review    │
│            ┆         ┆       ┆            ┆   ┆            ┆ N,UC       ┆            ┆ before    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ using     │
│ Cabin      ┆ String  ┆ 31    ┆ 125        ┆ … ┆ ✓ OK       ┆ MM         ┆ 89.830508  ┆ Minor     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ issues -  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ review    │
│ Embarked   ┆ String  ┆ 155   ┆ 1          ┆ … ┆ ✓ OK       ┆ -          ┆ 100.0      ┆ Good for  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
└────────────┴─────────┴───────┴────────────┴───┴────────────┴────────────┴────────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x12f1b6500>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center'

### 2.13 Formatting parameters (GT output only)

| Parameter | Default | Description |
|-----------|---------|-------------|
| `decimals` | `2` | Number of decimal places |
| `sep_mark` | `','` | Thousands separator |
| `dec_mark` | `'.'` | Decimal separator |
| `compact` | `False` | `True` -> '10K' instead of '10,000' |
| `pattern` | `None` | Wraps values, e.g. `'({x})'` or `'[{x}]'` |

In [31]:
# decimals=4
ps.xray(df_diabetes, decimals=4, title="4 decimal places")

GT(_tbl_data=shape: (9, 16)
┌─────────────┬─────────┬───────┬────────────┬───┬─────────────┬────────────┬────────┬─────────────┐
│ Column      ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Pct_Missing ┆ N_Outliers ┆ skew   ┆ Distributio │
│ ---         ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---         ┆ ---        ┆ ---    ┆ n_Plot      │
│ str         ┆ str     ┆ i64   ┆ i64        ┆   ┆ f64         ┆ i64        ┆ f64    ┆ ---         │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ list[f64]   │
╞═════════════╪═════════╪═══════╪════════════╪═══╪═════════════╪════════════╪════════╪═════════════╡
│ Pregnancies ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 4          ┆ 0.9    ┆ [246.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 103.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Glucose     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 5          ┆ 0.173  ┆ [5.0, 0.0,  │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 35.0]     │
│ BloodPressu ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 45         ┆ -1.84  ┆ [35.0, 0.0, │
│ re          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 2.0]      │
│ SkinThickne ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 1          ┆ 0.109  ┆ [231.0,     │
│ ss          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 55.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Insulin     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 35         ┆ 2.268  ┆ [456.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 148.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ BMI         ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 19         ┆ -0.428 ┆ [11.0, 0.0, │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 1.0]      │
│ DiabetesPed ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 29         ┆ 1.916  ┆ [266.0,     │
│ igreeFuncti ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 205.0, …    │
│ on          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 3.0]        │
│ Age         ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 9          ┆ 1.127  ┆ [267.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 150.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Outcome     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.634  ┆ [500.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, …      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 268.0]      │
└─────────────┴─────────┴───────┴────────────┴───┴─────────────┴────────────┴────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x12f1f2860>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTy

In [32]:
# compact=True -> uses suffixes like '10K', '1.5M'
ps.xray(df_diabetes, compact=True, title="Compact number format")

GT(_tbl_data=shape: (9, 16)
┌─────────────┬─────────┬───────┬────────────┬───┬─────────────┬────────────┬────────┬─────────────┐
│ Column      ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Pct_Missing ┆ N_Outliers ┆ skew   ┆ Distributio │
│ ---         ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---         ┆ ---        ┆ ---    ┆ n_Plot      │
│ str         ┆ str     ┆ i64   ┆ i64        ┆   ┆ f64         ┆ i64        ┆ f64    ┆ ---         │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ list[f64]   │
╞═════════════╪═════════╪═══════╪════════════╪═══╪═════════════╪════════════╪════════╪═════════════╡
│ Pregnancies ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 4          ┆ 0.9    ┆ [246.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 103.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Glucose     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 5          ┆ 0.173  ┆ [5.0, 0.0,  │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 35.0]     │
│ BloodPressu ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 45         ┆ -1.84  ┆ [35.0, 0.0, │
│ re          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 2.0]      │
│ SkinThickne ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 1          ┆ 0.109  ┆ [231.0,     │
│ ss          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 55.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Insulin     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 35         ┆ 2.268  ┆ [456.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 148.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ BMI         ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 19         ┆ -0.428 ┆ [11.0, 0.0, │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 1.0]      │
│ DiabetesPed ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 29         ┆ 1.916  ┆ [266.0,     │
│ igreeFuncti ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 205.0, …    │
│ on          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 3.0]        │
│ Age         ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 9          ┆ 1.127  ┆ [267.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 150.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Outcome     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.634  ┆ [500.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, …      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 268.0]      │
└─────────────┴─────────┴───────┴────────────┴───┴─────────────┴────────────┴────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x12f225de0>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTy

In [33]:
# European-style: sep_mark=' ', dec_mark=','
ps.xray(df_diabetes, sep_mark=' ', dec_mark=',', title="European number format")

GT(_tbl_data=shape: (9, 16)
┌─────────────┬─────────┬───────┬────────────┬───┬─────────────┬────────────┬────────┬─────────────┐
│ Column      ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Pct_Missing ┆ N_Outliers ┆ skew   ┆ Distributio │
│ ---         ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---         ┆ ---        ┆ ---    ┆ n_Plot      │
│ str         ┆ str     ┆ i64   ┆ i64        ┆   ┆ f64         ┆ i64        ┆ f64    ┆ ---         │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ list[f64]   │
╞═════════════╪═════════╪═══════╪════════════╪═══╪═════════════╪════════════╪════════╪═════════════╡
│ Pregnancies ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 4          ┆ 0.9    ┆ [246.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 103.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Glucose     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 5          ┆ 0.173  ┆ [5.0, 0.0,  │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 35.0]     │
│ BloodPressu ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 45         ┆ -1.84  ┆ [35.0, 0.0, │
│ re          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 2.0]      │
│ SkinThickne ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 1          ┆ 0.109  ┆ [231.0,     │
│ ss          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 55.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Insulin     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 35         ┆ 2.268  ┆ [456.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 148.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ BMI         ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 19         ┆ -0.428 ┆ [11.0, 0.0, │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 1.0]      │
│ DiabetesPed ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 29         ┆ 1.916  ┆ [266.0,     │
│ igreeFuncti ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 205.0, …    │
│ on          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 3.0]        │
│ Age         ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 9          ┆ 1.127  ┆ [267.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 150.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Outcome     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.634  ┆ [500.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, …      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 268.0]      │
└─────────────┴─────────┴───────┴────────────┴───┴─────────────┴────────────┴────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x12f0a7df0>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTy

In [34]:
# pattern='[{x}]' -> wraps each formatted value in brackets
ps.xray(df_diabetes, pattern='[{x}]', title="Values wrapped in brackets")

GT(_tbl_data=shape: (9, 16)
┌─────────────┬─────────┬───────┬────────────┬───┬─────────────┬────────────┬────────┬─────────────┐
│ Column      ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Pct_Missing ┆ N_Outliers ┆ skew   ┆ Distributio │
│ ---         ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---         ┆ ---        ┆ ---    ┆ n_Plot      │
│ str         ┆ str     ┆ i64   ┆ i64        ┆   ┆ f64         ┆ i64        ┆ f64    ┆ ---         │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ list[f64]   │
╞═════════════╪═════════╪═══════╪════════════╪═══╪═════════════╪════════════╪════════╪═════════════╡
│ Pregnancies ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 4          ┆ 0.9    ┆ [246.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 103.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Glucose     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 5          ┆ 0.173  ┆ [5.0, 0.0,  │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 35.0]     │
│ BloodPressu ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 45         ┆ -1.84  ┆ [35.0, 0.0, │
│ re          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 2.0]      │
│ SkinThickne ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 1          ┆ 0.109  ┆ [231.0,     │
│ ss          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 55.0, …     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Insulin     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 35         ┆ 2.268  ┆ [456.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 148.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ BMI         ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 19         ┆ -0.428 ┆ [11.0, 0.0, │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ … 1.0]      │
│ DiabetesPed ┆ Float64 ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 29         ┆ 1.916  ┆ [266.0,     │
│ igreeFuncti ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 205.0, …    │
│ on          ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 3.0]        │
│ Age         ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 9          ┆ 1.127  ┆ [267.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 150.0, …    │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 1.0]        │
│ Outcome     ┆ Int64   ┆ 768   ┆ 0          ┆ … ┆ 0.0         ┆ 0          ┆ 0.634  ┆ [500.0,     │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 0.0, …      │
│             ┆         ┆       ┆            ┆   ┆             ┆            ┆        ┆ 268.0]      │
└─────────────┴─────────┴───────┴────────────┴───┴─────────────┴────────────┴────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x12f22cc40>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='null_count', type=<ColInfoTypeEnum.default: 1>, column_label='null_count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center', column_width=None), ColInfo(var='std', type=<ColInfoTypeEnum.default: 1>, column_label='std', column_align='center', column_width=None), ColInfo(var='Min', type=<ColInfoTy

---
## 3 | `clean_column_names()`

```python
ps.clean_column_names(
    df,
    *,
    case       = 'lower',   # 'lower' | 'upper'
    ascii_only = True,      # True -> strip non-ASCII | False -> keep unicode
    dedupe     = True,      # True -> append _1, _2 to duplicates | False -> allow dupes
)
```

In [35]:
df_messy = pl.DataFrame({
    'First Name':   ['Alice', 'Bob'],
    'Last-Name':    ['Smith', 'Jones'],
    'Age (years)':  [30, 25],
    'Price €':      [80.5, 81.2],
})

# case='lower' (default), ascii_only=True (default), dedupe=True (default)
print("Original:", df_messy.columns)
cleaned = ps.clean_column_names(df_messy)
print("Default :", cleaned.columns)

Original: ['First Name', 'Last-Name', 'Age (years)', 'Price €']
Default : ['first_name', 'last_name', 'age_years', 'price_']


In [36]:
# case='upper'
upper = ps.clean_column_names(df_messy, case='upper')
print("Upper:", upper.columns)

Upper: ['FIRST_NAME', 'LAST_NAME', 'AGE_YEARS', 'PRICE_']


In [37]:
# ascii_only=False -> keep non-ASCII characters
keep_uni = ps.clean_column_names(df_messy, ascii_only=False)
print("Keep unicode:", keep_uni.columns)

Keep unicode: ['first_name', 'last_name', 'age_years', 'price_']


In [39]:
# dedupe=True (default) with actual duplicates
df_dupes = pl.DataFrame([
    pl.Series('Col A', [1, 2]),
    pl.Series('Col B', [3, 4]),    # duplicate name
    pl.Series('Col C', [5, 6]),    # another duplicate
])
print("Original:", df_dupes.columns)
deduped = ps.clean_column_names(df_dupes, dedupe=True)
print("Deduped :", deduped.columns)  # -> col_a, col_a_1, col_a_2

Original: ['Col A', 'Col B', 'Col C']
Deduped : ['col_a', 'col_b', 'col_c']


In [40]:
# dedupe=False -> no suffix for duplicate names
no_ded = ps.clean_column_names(df_dupes, dedupe=False)
print("No dedupe:", no_ded.columns)

No dedupe: ['col_a', 'col_b', 'col_c']


---
## 4 | `convert_datatypes()`

```python
ps.convert_datatypes(
    df,
    *,
    max_cardinality       = 20,    # int - max unique values for String -> Categorical
    categorical_threshold = 0.5,   # float - unique/total ratio threshold
    str_to_cat            = True,  # True -> convert eligible strings | False -> skip
    downcast_ints         = True,  # True -> Int64 -> Int8/16/32 | False -> skip
    downcast_floats       = True,  # True -> Float64 -> Float32 | False -> skip
)
```

In [41]:
df_types = pl.DataFrame({
    'big_int':    [1, 2, 3, 4, 5],                     # Int64 -> UInt8
    'big_float':  [1.1, 2.2, 3.3, 4.4, 5.5],          # Float64 -> Float32
    'category':   ['cat', 'dog', 'cat', 'dog', 'cat'], # String -> Categorical
    'high_card':  ['a', 'b', 'c', 'd', 'e'],           # 100% unique -> stays String
})

print("Before:", {c: str(d) for c, d in zip(df_types.columns, df_types.dtypes)})

# All defaults: str_to_cat=True, downcast_ints=True, downcast_floats=True
opt = ps.convert_datatypes(df_types)
print("After :", {c: str(d) for c, d in zip(opt.columns, opt.dtypes)})

Before: {'big_int': 'Int64', 'big_float': 'Float64', 'category': 'String', 'high_card': 'String'}
After : {'big_int': 'UInt8', 'big_float': 'Float32', 'category': 'Categorical', 'high_card': 'String'}


In [44]:
# str_to_cat=False -> don't convert strings
# downcast_ints=False -> keep original integer types
# downcast_floats=False -> keep Float64
no_opt = ps.convert_datatypes(
    df_types,
    str_to_cat=False,
    downcast_ints=False,
    downcast_floats=False,
)
print("No optimisation:", {c: str(d) for c, d in zip(no_opt.columns, no_opt.dtypes)})

No optimisation: {'big_int': 'Int64', 'big_float': 'Float64', 'category': 'String', 'high_card': 'String'}


In [45]:
# max_cardinality=2 -> only convert if <= 2 unique values
# categorical_threshold=0.3 -> and unique/total <= 30%
strict_cat = ps.convert_datatypes(
    df_types,
    max_cardinality=2,
    categorical_threshold=0.3,
)
print("Strict cat:", {c: str(d) for c, d in zip(strict_cat.columns, strict_cat.dtypes)})

Strict cat: {'big_int': 'UInt8', 'big_float': 'Float32', 'category': 'String', 'high_card': 'String'}


---
## 5 | `drop_missing()`

```python
ps.drop_missing(
    df,
    *,
    axis   = 'rows',    # 'rows' | 'columns'
    thresh = None,      # None (drop any null) | float 0.0-1.0 (min non-null fraction)
    subset = None,      # None (all columns) | ['col_a', 'col_b'] (specific columns)
)
```

In [46]:
df_miss = pl.DataFrame({
    'a': [1,    None, 3,    None, 5],
    'b': [None, 2,    None, 4,    None],
    'c': [10,   20,   30,   40,   50],
    'd': [None, None, None, None, 1],
})
print(f"Original: {df_miss.shape}")
df_miss

Original: (5, 4)


a,b,c,d
i64,i64,i64,i64
1,null,10,null
null,2,20,null
3,null,30,null
null,4,40,null
5,null,50,1


In [47]:
# axis='rows', thresh=None -> drop rows with ANY null
r1 = ps.drop_missing(df_miss, axis='rows', thresh=None)
print(f"axis='rows', thresh=None:  {r1.shape}")

# axis='rows', thresh=0.5 -> keep rows where >= 50% of columns are non-null
r2 = ps.drop_missing(df_miss, axis='rows', thresh=0.5)
print(f"axis='rows', thresh=0.5:   {r2.shape}")

# axis='rows', subset=['a'] -> only check column 'a' for nulls
r3 = ps.drop_missing(df_miss, axis='rows', subset=['a'])
print(f"axis='rows', subset=['a']: {r3.shape}")

# axis='rows', thresh=0.5, subset=['a','b'] -> combined
r4 = ps.drop_missing(df_miss, axis='rows', thresh=0.5, subset=['a', 'b'])
print(f"axis='rows', thresh=0.5, subset=['a','b']: {r4.shape}")

axis='rows', thresh=None:  (0, 4)
axis='rows', thresh=0.5:   (5, 4)
axis='rows', subset=['a']: (3, 4)
axis='rows', thresh=0.5, subset=['a','b']: (5, 4)


In [48]:
# axis='columns', thresh=None -> keep only columns with ZERO nulls
c1 = ps.drop_missing(df_miss, axis='columns', thresh=None)
print(f"axis='columns', thresh=None: {c1.shape} -> kept: {c1.columns}")

# axis='columns', thresh=0.5 -> keep columns where >= 50% are non-null
c2 = ps.drop_missing(df_miss, axis='columns', thresh=0.5)
print(f"axis='columns', thresh=0.5:  {c2.shape} -> kept: {c2.columns}")

# axis='columns', thresh=0.8 -> keep columns where >= 80% are non-null
c3 = ps.drop_missing(df_miss, axis='columns', thresh=0.8)
print(f"axis='columns', thresh=0.8:  {c3.shape} -> kept: {c3.columns}")

axis='columns', thresh=None: (5, 1) -> kept: ['c']
axis='columns', thresh=0.5:  (5, 2) -> kept: ['a', 'c']
axis='columns', thresh=0.8:  (5, 1) -> kept: ['c']


---
## 6 | `data_cleaning()`

```python
ps.data_cleaning(
    df,
    *,
    drop_missing_thresh   = 0.9,    # float - keep columns with >= this non-null fraction
    optimize_dtypes       = True,    # True | False
    remove_duplicates     = True,    # True | False
    outlier_method        = 'iqr',   # 'iqr' | 'zscore' | None
    outlier_threshold     = 1.5,     # float - IQR multiplier or z-score cutoff
    categorical_threshold = 0.5,     # float - passed to convert_datatypes
    max_cardinality       = 50,      # int   - passed to convert_datatypes
)
```

In [49]:
np.random.seed(42)
df_dirty = pl.DataFrame({
    'value':    list(np.random.normal(50, 10, 95)) + [500, -500, 50, 50, 50],
    'category': np.random.choice(['A', 'B', 'C'], 100).tolist(),
    'sparse':   [None]*92 + list(range(8)),
})

# All defaults
df_clean = ps.data_cleaning(df_dirty)

Starting data cleaning: (100, 3)
After dropping sparse columns: (100, 2)
Removed 1 duplicate rows
Data types optimized
Data cleaning complete: (99, 2)


In [50]:
# outlier_method='zscore', outlier_threshold=2.0
df_zs = ps.data_cleaning(
    df_dirty,
    drop_missing_thresh=0.5,
    optimize_dtypes=True,
    remove_duplicates=True,
    outlier_method='zscore',
    outlier_threshold=2.0,
    categorical_threshold=0.4,
    max_cardinality=20,
)

Starting data cleaning: (100, 3)
After dropping sparse columns: (100, 2)
Removed 1 duplicate rows
Data types optimized
Data cleaning complete: (99, 2)


In [51]:
# outlier_method=None -> skip outlier handling
# optimize_dtypes=False, remove_duplicates=False -> minimal cleaning
df_min = ps.data_cleaning(
    df_dirty,
    outlier_method=None,
    optimize_dtypes=False,
    remove_duplicates=False,
)

Starting data cleaning: (100, 3)
After dropping sparse columns: (100, 2)
Data cleaning complete: (100, 2)


---
## 7 | `corr_heatmap()`

```python
ps.corr_heatmap(
    df,
    columns   = None,       # Sequence[str] | None - specific columns to include
    *,
    split     = None,       # None | 'pos' | 'neg' | 'high' | 'low'
    threshold = 0.0,        # float 0-1 - correlation filter threshold
    target    = None,       # str | None - target column for 1-vs-all correlation
    method    = 'pearson',  # 'pearson' | 'spearman'
    annotate  = True,       # True | False - show values on cells
    width     = None,       # int | None - plot width in pixels
    height    = None,       # int | None - plot height in pixels
    backend   = 'plotly',   # 'plotly' | 'altair'
)
```

**`split` options:**
| Value | Keeps |
|-------|-------|
| `None` | All correlations |
| `'pos'` | Only positive correlations > threshold |
| `'neg'` | Only negative correlations < -threshold |
| `'high'` | Only \|corr\| > threshold (default threshold becomes 0.3) |
| `'low'` | Only \|corr\| < threshold |

In [52]:
# Default: all numeric columns, method='pearson', backend='plotly', annotate=True
ps.corr_heatmap(df_diabetes)

In [53]:
# method='spearman' - rank-based correlation
ps.corr_heatmap(df_diabetes, method='spearman')

In [54]:
# target='Outcome' - 1-row heatmap: all features vs target
ps.corr_heatmap(df_diabetes, target='Outcome')

In [55]:
# target + method='spearman'
ps.corr_heatmap(df_diabetes, target='Outcome', method='spearman')

In [56]:
# split='high', threshold=0.3 -> only show |corr| > 0.3
ps.corr_heatmap(df_diabetes, split='high', threshold=0.3)

In [57]:
# split='pos', threshold=0.2 -> only positive correlations > 0.2
ps.corr_heatmap(df_diabetes, split='pos', threshold=0.2)

In [58]:
# split='neg', threshold=0.1 -> only negative correlations < -0.1
ps.corr_heatmap(df_diabetes, split='neg', threshold=0.1)

In [59]:
# split='low', threshold=0.3 -> only weak correlations |corr| < 0.3
ps.corr_heatmap(df_diabetes, split='low', threshold=0.3)

In [60]:
# columns=['Glucose', 'BMI', 'Age', 'Outcome'] -> specific columns only
ps.corr_heatmap(df_diabetes, columns=['Glucose', 'BMI', 'Age', 'Outcome'])

In [61]:
# annotate=False -> no text values on cells
ps.corr_heatmap(df_diabetes, annotate=False)

In [62]:
# width=900, height=700 -> custom dimensions
ps.corr_heatmap(df_diabetes, width=900, height=700)

In [63]:
# backend='altair'
ps.corr_heatmap(df_diabetes, backend='altair')

alt.LayerChart(...)

In [64]:
# backend='altair' with target
ps.corr_heatmap(df_diabetes, target='Outcome', backend='altair')

alt.LayerChart(...)

---
## 8 | `dist_plot()`

```python
ps.dist_plot(
    df,
    column  = None,       # str | None - column name (default: first numeric)
    *,
    bins    = 30,         # int - number of histogram bins
    width   = None,       # int | None - plot width in pixels
    height  = None,       # int | None - plot height in pixels
    backend = 'plotly',   # 'plotly' | 'altair'
)
```

In [65]:
# Default: column=None (first numeric), bins=30, backend='plotly'
ps.dist_plot(df_diabetes)

In [66]:
# column='Glucose', bins=50
ps.dist_plot(df_diabetes, column='Glucose', bins=50)

In [67]:
# column='BMI', bins=15
ps.dist_plot(df_diabetes, column='BMI', bins=15)

In [68]:
# width=900, height=400
ps.dist_plot(df_diabetes, column='Insulin', bins=40, width=900, height=400)

In [69]:
# backend='altair'
ps.dist_plot(df_diabetes, column='Age', bins=20, backend='altair')

alt.Chart(...)

In [70]:
# backend='altair' with custom width/height
ps.dist_plot(df_diabetes, column='Glucose', bins=30, width=600, height=300, backend='altair')

alt.Chart(...)

---
## 9 | `missingval_plot()`

```python
ps.missingval_plot(
    df,
    *,
    sort      = 'desc',    # 'desc' | 'asc' | 'none'
    normalize = False,     # False -> x-axis = count | True -> x-axis = percentage
    width     = None,      # int | None
    height    = None,      # int | None
    backend   = 'plotly',  # 'plotly' | 'altair'
)
```

In [71]:
# Default: sort='desc', normalize=False (absolute counts), backend='plotly'
ps.missingval_plot(df_titanic)

In [72]:
# normalize=True -> x-axis shows share (0-1) with percentage labels
ps.missingval_plot(df_titanic, normalize=True)

In [73]:
# sort='asc' -> least missing first
ps.missingval_plot(df_titanic, sort='asc')

In [74]:
# sort='none' -> original column order
ps.missingval_plot(df_titanic, sort='none')

In [75]:
# width=800, height=400
ps.missingval_plot(df_titanic, width=800, height=400)

In [76]:
# backend='altair', normalize=False
ps.missingval_plot(df_titanic, backend='altair')

alt.LayerChart(...)

In [77]:
# backend='altair', normalize=True
ps.missingval_plot(df_titanic, normalize=True, backend='altair')

alt.LayerChart(...)

---
## 10 | `cat_plot()`

```python
ps.cat_plot(
    df,
    *,
    top     = 10,       # int >= 0 - most frequent categories to show
    bottom  = 10,       # int >= 0 - least frequent categories to show
    width   = None,     # int | None
    height  = None,     # int | None
    backend = 'plotly', # 'plotly' | 'altair'
)
```

Automatically detects String/Categorical columns. Blue bars = top categories, red = bottom.

In [78]:
# Default: top=10, bottom=10, backend='plotly'
ps.cat_plot(df_titanic)

In [79]:
# top=3, bottom=0 -> only 3 most frequent, no bottom
ps.cat_plot(df_titanic, top=3, bottom=0)

In [80]:
# top=5, bottom=5
ps.cat_plot(df_titanic, top=5, bottom=5)

In [81]:
# top=0, bottom=5 -> only least frequent categories
ps.cat_plot(df_titanic, top=0, bottom=5)

In [82]:
# width=1200, height=500
ps.cat_plot(df_titanic, top=5, bottom=0, width=1200, height=500)

In [83]:
# backend='altair'
ps.cat_plot(df_titanic, top=5, bottom=3, backend='altair')

alt.Chart(...)

---
## 11 | `corr_plot()`

```python
ps.corr_plot(
    df,
    columns     = None,       # list[str] | None - specific columns
    *,
    method      = 'pearson',  # 'pearson' | 'spearman'
    interactive = True,       # True -> forces plotly | False -> uses backend
    clustered   = False,      # True -> hierarchical clustering (requires scipy)
    width       = None,       # int | None
    height      = None,       # int | None
    backend     = 'plotly',   # 'plotly' | 'altair'
)
```

In [84]:
# Default: all numeric, method='pearson', interactive=True, backend='plotly'
ps.corr_plot(df_diabetes)

In [85]:
# method='spearman'
ps.corr_plot(df_diabetes, method='spearman')

In [86]:
# clustered=True -> reorders columns by correlation similarity (requires scipy)
ps.corr_plot(df_diabetes, clustered=True)

In [87]:
# clustered=True with method='spearman'
ps.corr_plot(df_diabetes, method='spearman', clustered=True)

In [88]:
# columns=['Glucose', 'BMI', 'Age', 'Insulin'] -> specific columns
ps.corr_plot(df_diabetes, columns=['Glucose', 'BMI', 'Age', 'Insulin'])

In [89]:
# interactive=False, backend='altair'
ps.corr_plot(df_diabetes, interactive=False, backend='altair')

alt.Chart(...)

In [90]:
# width=900, height=700
ps.corr_plot(df_diabetes, width=900, height=700)

---
## 12 | `save_fig()`

```python
ps.save_fig(
    obj,           # Plotly Figure | Altair Chart | Matplotlib Figure
    path,          # str - output filepath (extension determines format)
    *,
    scale = 1.0,   # float - scaling factor for Plotly static images
)
```

| Backend | Supported formats |
|---------|-------------------|
| Plotly | `.html` (no deps), `.png`/`.pdf`/`.svg` (requires kaleido) |
| Altair | `.html`, `.png`, `.svg` (may require vl-convert) |

In [91]:
import tempfile, os

# Save a Plotly figure as HTML
fig_plotly = ps.corr_heatmap(df_diabetes, backend='plotly')

with tempfile.TemporaryDirectory() as tmpdir:
    html_path = os.path.join(tmpdir, 'corr_heatmap.html')
    ps.save_fig(fig_plotly, html_path)             # scale not used for HTML
    print(f"Plotly HTML: {os.path.getsize(html_path):,} bytes")

# Save with scale parameter (for static image export, requires kaleido)
# ps.save_fig(fig_plotly, 'corr_heatmap.png', scale=2.0)

Plotly HTML: 10,820 bytes


---
## 13 | Full Pipeline Example

Putting it all together: load -> clean -> inspect -> visualise.

In [92]:
# 1. Load
df = titanic()

# 2. Clean column names
df = ps.clean_column_names(df)
print("Columns:", df.columns)

# 3. Full xray with model usability
ps.xray(
    df,
    include='all',
    expanded=True,
    corr_target='survived',
    model_usability=True,
    title="Full Titanic Pipeline"
)

Columns: ['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']


GT(_tbl_data=shape: (12, 37)
┌────────────┬─────────┬───────┬────────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ Column     ┆ Dtype   ┆ Count ┆ null_count ┆ … ┆ Quality_Fl ┆ Usability_ ┆ Usability_ ┆ Recommend │
│ ---        ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ag         ┆ Flags      ┆ Score      ┆ ation     │
│ str        ┆ str     ┆ i64   ┆ i64        ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│            ┆         ┆       ┆            ┆   ┆ str        ┆ str        ┆ f64        ┆ str       │
╞════════════╪═════════╪═══════╪════════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ passengeri ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ ⚠ SHAKY    ┆ ID,NN,UC   ┆ 77.966102  ┆ Drop -    │
│ d          ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ likely an │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ ID column │
│ survived   ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ ✓ OK       ┆ BN,NN,UC   ┆ 88.135593  ┆ Minor     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ issues -  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ review    │
│ pclass     ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ ✓ OK       ┆ NN,UC      ┆ 91.525424  ┆ Good for  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ name       ┆ String  ┆ 156   ┆ 0          ┆ … ┆ ✓ OK       ┆ ID         ┆ 86.440678  ┆ Drop -    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ likely an │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ ID column │
│ sex        ┆ String  ┆ 156   ┆ 0          ┆ … ┆ ✓ OK       ┆ BN         ┆ 96.610169  ┆ Good for  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ …          ┆ …       ┆ …     ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …         │
│ parch      ┆ Int64   ┆ 156   ┆ 0          ┆ … ┆ ⚠ SHAKY    ┆ EK,EO,NN,U ┆ 76.271186  ┆ Minor     │
│            ┆         ┆       ┆            ┆   ┆            ┆ C          ┆            ┆ issues -  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ review    │
│ ticket     ┆ String  ┆ 156   ┆ 0          ┆ … ┆ ✓ OK       ┆ -          ┆ 100.0      ┆ Good for  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
│ fare       ┆ Float64 ┆ 156   ┆ 0          ┆ … ┆ ⚠ SHAKY    ┆ EK,EO,ES,N ┆ 69.491525  ┆ Review    │
│            ┆         ┆       ┆            ┆   ┆            ┆ N,UC       ┆            ┆ before    │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ using     │
│ cabin      ┆ String  ┆ 31    ┆ 125        ┆ … ┆ ✓ OK       ┆ MM         ┆ 89.830508  ┆ Minor     │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ issues -  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ review    │
│ embarked   ┆ String  ┆ 155   ┆ 1          ┆ … ┆ ✓ OK       ┆ -          ┆ 100.0      ┆ Good for  │
│            ┆         ┆       ┆            ┆   ┆            ┆            ┆            ┆ modeling  │
└────────────┴─────────┴───────┴────────────┴───┴────────────┴────────────┴────────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x15cbd6950>, _boxhead=Boxhead([ColInfo(var='Column', type=<ColInfoTypeEnum.default: 1>, column_label='Column', column_align='left', column_width=None), ColInfo(var='Dtype', type=<ColInfoTypeEnum.default: 1>, column_label='Dtype', column_align='center', column_width=None), ColInfo(var='Count', type=<ColInfoTypeEnum.default: 1>, column_label='Count', column_align='center', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='center'

In [93]:
# 4. Missing values (absolute counts)
ps.missingval_plot(df)

In [94]:
# 5. Target correlation
ps.corr_heatmap(df, target='survived', method='spearman')

In [95]:
# 6. Distribution of fare
ps.dist_plot(df, column='fare', bins=40)

In [96]:
# 7. Categorical analysis
ps.cat_plot(df, top=5, bottom=3)

In [97]:
# 8. Clustered correlation plot
ps.corr_plot(df, clustered=True, method='spearman')

scipy clustering unavailable (ValueError), skipping clustering


In [98]:
print("Demo complete!")

Demo complete!
